In [0]:
dbutils.widgets.dropdown(name="environment", defaultValue="dev",
                         choices=["dev","qa","prd"], label="select Environment")
env = dbutils.widgets.get("environment")
print(f"Environment: {env}")

catalog = f"saleslake_{env}"
bronzeSchema = f"bronze_schema_{env}"

targetTable    = f"{catalog}.{bronzeSchema}.rawProduct"
# srcFileLoc  = f"s3://saleslake-748005667258-eu-north-1-an/saleslake/{env}/source_files/product/"
srcFileLoc=f"/Volumes/{catalog}/{bronzeSchema}/vol_saleslake_src_files_{env}/daily_product/"
print(f"Target table : {targetTable}")
print(f"Source files : {srcFileLoc}")

In [0]:
spark.sql(f"""
COPY INTO {targetTable}
FROM (
    SELECT product_id, sku, product_name, category, sub_category, brand, supplier, unit_cost, list_price, launch_date, status,
           current_timestamp() AS ingest_ts
    FROM "{srcFileLoc}"
)
FILEFORMAT = CSV
FORMAT_OPTIONS ('header' = 'true')
COPY_OPTIONS  ('mergeSchema' = 'false')
""")

print(f"Bronze load complete for {targetTable}")



In [0]:
spark.sql(f"SELECT COUNT(*) AS row_count FROM {targetTable}").display()